# 03 - Feature engineering

Take the weekly targets (`all_cities_weekly_targets.csv`) + the weekly feature table (`all_cities_weekly_features.csv`) and produce the modelling matrix:

* Inner-join on `(adm3_pcode, date)`.
* Per-city target lags: both cases and deaths use only the **4-week lag** (the 1–3 week lags are near-perfectly autocorrelated and were dropped to avoid trivially dominant features).
* Calendar features: `year`, `month`, `week_of_year`, sin/cos of week.
* One-hot encode `adm3_pcode`.
* Drop any feature column that ended up >50% NaN on the joined frame.
* Emit `regression_modeling_table.csv` plus `feature_schema.json` (per-feature p1/p50/p99 + default = median). The JSON is the contract for the Streamlit UI.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "data").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
PROC_DIR = REPO_ROOT / "data" / "processed"
EDA_DIR  = PROC_DIR / "eda"

TARGET_CITIES = sorted(json.loads((EDA_DIR / "target_adm3_pcodes.json").read_text()))

## 1. Load target + feature blocks; inner-join on `(adm3_pcode, date)`

In [2]:
target = pd.read_csv(PROC_DIR / "health" / "all_cities_weekly_targets.csv", parse_dates=["date"])
features = pd.read_csv(PROC_DIR / "all_cities_weekly_features.csv", parse_dates=["date"])
print("target  :", target.shape)
print("features:", features.shape)

df = target.merge(features, on=["adm3_pcode", "date"], how="inner").sort_values(["adm3_pcode", "date"]).reset_index(drop=True)
print("joined  :", df.shape)

target  : (10656, 6)
features: (12528, 222)
joined  : (10656, 226)


## 2. Sparsity filter (drop columns >50% NaN on the joined frame)

In [3]:
META = {"adm3_pcode", "date", "all_cause_cases", "all_cause_deaths", "has_pidsr_report", "has_psa_report"}
feature_cols = [c for c in df.columns if c not in META]
miss = df[feature_cols].isna().mean()
drop_cols = miss[miss > 0.5].index.tolist()
print(f"dropping {len(drop_cols)} of {len(feature_cols)} features (>50% NaN)")
if drop_cols:
    print("  columns to drop:", drop_cols)
df = df.drop(columns=drop_cols)
feature_cols = [c for c in df.columns if c not in META]
print("remaining feature cols:", len(feature_cols))

dropping 16 of 220 features (>50% NaN)
  columns to drop: ['tave_downscaled', 'mobile_mean_avg_d_kbps_mean', 'mobile_mean_avg_u_kbps_mean', 'mobile_mean_avg_lat_ms_mean', 'mobile_mean_num_tests_mean', 'mobile_mean_num_devices_mean', 'fixed_mean_avg_d_kbps_mean', 'fixed_mean_avg_u_kbps_mean', 'fixed_mean_avg_lat_ms_mean', 'fixed_mean_num_tests_mean', 'fixed_mean_num_devices_mean', 'rwi_max', 'rwi_mean', 'rwi_median', 'rwi_min', 'rwi_std']
remaining feature cols: 204


## 3. Per-city lags of the two targets

Both cases and deaths use only the **4-week lag**. The 1–3 week lags are near-perfectly autocorrelated with the current week's value, making them trivially dominant features rather than meaningful predictors. The 4-week anchor gives the model a medium-range trend signal without the short-range leakage problem.

In [4]:
LAGS_BY_TARGET = {
    "all_cause_cases":  [4],
    "all_cause_deaths": [4],
}
for col, lags in LAGS_BY_TARGET.items():
    for k in lags:
        df[f"{col}_lag{k}wk"] = df.groupby("adm3_pcode")[col].shift(k)

lag_cols = [c for c in df.columns if c.startswith(("all_cause_cases_lag", "all_cause_deaths_lag"))]
print("lag columns added:", len(lag_cols), "->", lag_cols)
print("NaN in lag rows (will be dropped after split):")
print(df[lag_cols].isna().sum())

lag columns added: 2 -> ['all_cause_cases_lag4wk', 'all_cause_deaths_lag4wk']
NaN in lag rows (will be dropped after split):
all_cause_cases_lag4wk     48
all_cause_deaths_lag4wk    48
dtype: int64


/tmp/claude-1275385993/ipykernel_6138/3540806094.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{k}wk"] = df.groupby("adm3_pcode")[col].shift(k)
/tmp/claude-1275385993/ipykernel_6138/3540806094.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{k}wk"] = df.groupby("adm3_pcode")[col].shift(k)


## 4. Calendar features

Year/month/week-of-year as ints + sin/cos for week-of-year so tree models can pick up periodicity smoothly.

In [5]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
two_pi_w = 2 * np.pi * df["week_of_year"] / 52.0
df["week_sin"] = np.sin(two_pi_w)
df["week_cos"] = np.cos(two_pi_w)
print("calendar cols added: year, month, week_of_year, week_sin, week_cos")
df[["date", "year", "month", "week_of_year", "week_sin", "week_cos"]].head()

calendar cols added: year, month, week_of_year, week_sin, week_cos


/tmp/claude-1275385993/ipykernel_6138/4096166836.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["year"] = df["date"].dt.year
/tmp/claude-1275385993/ipykernel_6138/4096166836.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["month"] = df["date"].dt.month
/tmp/claude-1275385993/ipykernel_6138/4096166836.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) ins

,date,year,month,week_of_year,week_sin,week_cos
0,2005-12-26,2005,12,52,6.432491e-16,1.000000
1,2006-01-02,2006,1,1,1.205367e-01,0.992709
2,2006-01-09,2006,1,2,2.393157e-01,0.970942
3,2006-01-16,2006,1,3,3.546049e-01,0.935016
4,2006-01-23,2006,1,4,4.647232e-01,0.885456


## 5. One-hot encode `adm3_pcode`

In [6]:
city_dummies = pd.get_dummies(df["adm3_pcode"], prefix="city", dtype=int)
df = pd.concat([df, city_dummies], axis=1)
print("city dummies added:", list(city_dummies.columns))

city dummies added: ['city_PH015518000', 'city_PH034919000', 'city_PH050506000', 'city_PH063022000', 'city_PH072230000', 'city_PH083747000', 'city_PH097332000', 'city_PH104305000', 'city_PH112402000', 'city_PH137401000', 'city_PH137503000', 'city_PH137603000']


## 6. Build `feature_schema.json` for the Streamlit UI

For every numeric input column we save: dtype, p1 / p50 / p99 (used for slider bounds + default value). Categorical inputs (the city) get their list of legal values.

In [7]:
MODEL_TARGETS = ["all_cause_cases", "all_cause_deaths"]
EXCLUDE_FROM_FEATURES = set(MODEL_TARGETS) | {"has_pidsr_report", "has_psa_report", "date"}
feature_cols_final = [c for c in df.columns if c not in EXCLUDE_FROM_FEATURES and c != "adm3_pcode"]
print("final feature columns:", len(feature_cols_final))

schema = {
    "version": 1,
    "adm3_pcode": {"type": "categorical", "choices": TARGET_CITIES},
    "features": {},
}
for c in feature_cols_final:
    s = pd.to_numeric(df[c], errors="coerce").dropna()
    if len(s) == 0:
        continue
    p1, p50, p99 = (float(x) for x in np.nanpercentile(s, [1, 50, 99]))
    schema["features"][c] = {
        "dtype": str(df[c].dtype),
        "p1": p1,
        "p50": p50,
        "p99": p99,
        "is_lag": c.startswith(("all_cause_cases_lag", "all_cause_deaths_lag")),
        "is_calendar": c in {"year", "month", "week_of_year", "week_sin", "week_cos"},
        "is_city_dummy": c.startswith("city_"),
    }

out_schema = PROC_DIR / "feature_schema.json"
out_schema.write_text(json.dumps(schema, indent=2))
print("wrote", out_schema)
print("feature entries:", len(schema["features"]))

final feature columns: 223
wrote /Users/adonaisray.maclang/Desktop/morbidity-mortality-modelling/data/processed/feature_schema.json
feature entries: 223


## 7. Seasonal-median table for the Streamlit "Reset to seasonal median" button

Per `(adm3_pcode, week_of_year)`, the median value of every modelling feature. Streamlit reads this so users don't have to set 200+ inputs by hand.

In [8]:
GROUP_KEYS = ["adm3_pcode", "week_of_year"]
num_feature_cols = [
    c for c in feature_cols_final
    if c not in GROUP_KEYS and pd.api.types.is_numeric_dtype(df[c])
]
med = (
    df.groupby(GROUP_KEYS)[num_feature_cols]
    .median()
    .reset_index()
)
med_path = PROC_DIR / "seasonal_median_features.csv"
med.to_csv(med_path, index=False)
print("wrote", med_path, "shape:", med.shape)

wrote /Users/adonaisray.maclang/Desktop/morbidity-mortality-modelling/data/processed/seasonal_median_features.csv shape: (636, 224)


/tmp/claude-1275385993/ipykernel_6138/611313927.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()
/tmp/claude-1275385993/ipykernel_6138/611313927.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


## 8. Write the modelling table

In [9]:
out_path = PROC_DIR / "regression_modeling_table.csv"
df.to_csv(out_path, index=False)
print("wrote", out_path, "shape:", df.shape, "size_mb:", round(out_path.stat().st_size / 1e6, 1))

# Also persist the ordered list of input columns the model will expect at training time.
(PROC_DIR / "feature_columns.json").write_text(json.dumps(feature_cols_final, indent=2))
print("wrote feature_columns.json (", len(feature_cols_final), "cols )")

wrote /Users/adonaisray.maclang/Desktop/morbidity-mortality-modelling/data/processed/regression_modeling_table.csv shape: (10656, 229) size_mb: 25.1
wrote feature_columns.json ( 223 cols )


In [10]:
# Final feature columns
for col in feature_cols_final:
    print(col)


no2
co
so2
o3
pm10
pm25
tave
tmin
tmax
heat_index
pr
wind_speed
rh
solar_rad
uv_rad
tmin_downscaled
tmax_downscaled
pr_downscaled
ndvi
pr_norm
spi3
spi6
pnp
pop_count_total
pop_count_mean
pop_count_median
pop_count_stdev
pop_count_min
pop_count_max
pop_density_mean
pop_density_median
pop_density_stdev
pop_density_min
pop_density_max
avg_rad_min
avg_rad_max
avg_rad_mean
avg_rad_std
avg_rad_median
poi_count
atm_count
atm_nearest
bank_count
bank_nearest
college_count
college_nearest
community_centre_count
community_centre_nearest
convenience_count
convenience_nearest
fire_station_count
fire_station_nearest
kindergarten_count
kindergarten_nearest
lighthouse_count
lighthouse_nearest
market_place_count
market_place_nearest
park_count
park_nearest
police_count
police_nearest
school_count
school_nearest
shelter_count
shelter_nearest
supermarket_count
supermarket_nearest
telephone_count
telephone_nearest
town_hall_count
town_hall_nearest
university_count
university_nearest
clinic_count
clinic_n